In [ ]:
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결
conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    port=os.getenv("DB_PORT")
)

inserted_count = 0

try:
    cur = conn.cursor()

    # 📥 엑셀 데이터 로드 및 전처리
    df = pd.read_excel(r"C:\Users\wngus\Documents\SK하이닉스_정리본.xlsx")
    df['published_at'] = pd.to_datetime(df['published_at'])
    df['price_date'] = pd.to_datetime(df['price_date'])
    df['ticker'] = df['ticker'].astype(str).str.zfill(6)

    for _, row in df.iterrows():
        try:
            # ✅ ticker 존재 확인
            cur.execute("SELECT ticker FROM ticker WHERE ticker = %s", (row['ticker'],))
            if cur.fetchone() is None:
                print(f"⛔️ ticker 없음: {row['ticker']}")
                continue

            # ✅ stock_price 확인
            cur.execute("""
                SELECT 1 FROM stock_price
                WHERE ticker = %s AND price_date = %s
            """, (row['ticker'], row['price_date']))
            if cur.fetchone() is None:
                print(f"⛔️ stock_price 없음: {row['ticker']}, {row['price_date']}")
                continue

            # ✅ publisher 처리 (중복 안전하게)
            cur.execute("""
                WITH ins AS (
                    INSERT INTO publisher (name)
                    VALUES (%s)
                    ON CONFLICT (name) DO NOTHING
                    RETURNING publisher_id
                )
                SELECT publisher_id FROM ins
                UNION
                SELECT publisher_id FROM publisher WHERE name = %s;
            """, (row['publisher_name'], row['publisher_name']))
            publisher_id = cur.fetchone()[0]

            # ✅ news 삽입
            cur.execute("""
                INSERT INTO news (
                    ticker, price_date, publisher_id, title, summary, content, url,
                    sentiment, sentiment_score, published_at
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (url) DO NOTHING
                RETURNING news_id
            """, (
                row['ticker'],
                row['price_date'],
                publisher_id,
                row['title'],
                row.get('summary', ''),
                row.get('content', ''),
                row['url'],
                row.get('sentiment', None),
                float(row.get('sentiment_score', 0)),
                row['published_at']
            ))

            result = cur.fetchone()
            if result:
                news_id = result[0]
            else:
                cur.execute("SELECT news_id FROM news WHERE url = %s", (row['url'],))
                news_id = cur.fetchone()[0]

            # ✅ 키워드 처리 및 연결
            keywords = str(row.get('keywords', '')).split(',')
            for word in keywords:
                word = word.strip()
                if not word:
                    continue

                cur.execute("""
                    WITH ins AS (
                        INSERT INTO keyword (word)
                        VALUES (%s)
                        ON CONFLICT (word) DO NOTHING
                        RETURNING keyword_id
                    )
                    SELECT keyword_id FROM ins
                    UNION
                    SELECT keyword_id FROM keyword WHERE word = %s;
                """, (word, word))
                keyword_id = cur.fetchone()[0]

                cur.execute("""
                    INSERT INTO news_keyword (news_id, keyword_id)
                    VALUES (%s, %s)
                    ON CONFLICT DO NOTHING
                """, (news_id, keyword_id))

            inserted_count += 1

        except Exception as row_error:
            print(f"[❌ row 오류] {row.get('title', '제목없음')} - {row_error}")
            try:
                if conn and conn.closed == 0:
                    conn.rollback()
            except Exception as rollback_error:
                print(f"[⚠️ rollback 실패] {rollback_error}")
            continue

    conn.commit()
    print(f"✅ 총 {inserted_count}건 삽입 완료.")

except Exception as e:
    print(f"[❌ 전체 오류] {e}")

finally:
    try:
        if cur and not cur.closed:
            cur.close()
        if conn and conn.closed == 0:
            conn.close()
    except Exception as close_error:
        print(f"[⚠️ 종료 중 에러] {close_error}")

In [35]:
import os
import glob
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결 함수
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT")
    )

# 🔧 Normalize DataFrame to standard schema
STANDARD_COLUMNS = [
    'ticker', 'price_date', 'publisher_name', 'title',
    'content', 'url', 'published_at'
]

def normalize_df(df: pd.DataFrame, ticker: str = None) -> pd.DataFrame:
    df = df.copy()
    # ticker 채우기
    df['ticker'] = ticker
    # published_at 파싱
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['price_date'] = df['published_at'].dt.date
    # 부족한 컬럼 추가
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = None
    # 순서 맞추기
    return df[STANDARD_COLUMNS]

# 🛠 Publisher helper
def get_or_create_publisher(cur, name: str) -> int:
    cur.execute(
        """
        WITH ins AS (
          INSERT INTO publisher (name)
          VALUES (%s)
          ON CONFLICT (name) DO NOTHING
          RETURNING publisher_id
        )
        SELECT publisher_id FROM ins
        UNION
        SELECT publisher_id FROM publisher WHERE name = %s;
        """,
        (name, name)
    )
    return cur.fetchone()[0]

# 🚀 Main pipeline: read CSVs, normalize, insert
def pipeline_load_folder(folder_path: str):
    conn = get_db_connection()
    cur = conn.cursor()
    total_inserted = 0

    # .csv 파일만 읽기
    files = glob.glob(os.path.join(folder_path, '*.csv'))
    for file in files:
        # 파일명에서 ticker 추출
        fname = os.path.basename(file)
        parts = os.path.splitext(fname)[0].split('_')
        ticker = parts[-1] if parts[-1].isdigit() else None

        df_raw = pd.read_csv(file)
        df = normalize_df(df_raw, ticker=ticker)

        for _, row in df.iterrows():
            try:
                # ticker 유효성
                cur.execute("SELECT 1 FROM ticker WHERE ticker = %s", (row['ticker'],))
                if cur.fetchone() is None:
                    continue

                # stock_price 유효성
                cur.execute(
                    "SELECT 1 FROM stock_price WHERE ticker = %s AND price_date = %s",
                    (row['ticker'], row['price_date'])
                )
                if cur.fetchone() is None:
                    continue

                # publisher
                publisher_id = get_or_create_publisher(cur, row['publisher_name'] or 'Unknown')

                # news 삽입
                cur.execute(
                    """
                    INSERT INTO news (
                      ticker, price_date, publisher_id,
                      title, content, url, published_at
                    )
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (url) DO NOTHING RETURNING news_id;
                    """,
                    (
                        row['ticker'], row['price_date'], publisher_id,
                        row['title'], row['content'], row['url'],
                        row['published_at']
                    )
                )
                if cur.fetchone():
                    total_inserted += 1

            except Exception:
                conn.rollback()
                continue

        conn.commit()

    cur.close()
    conn.close()
    print(f"✅ 총 {total_inserted}건 삽입 완료")

if __name__ == '__main__':
    # csv 파일들이 모여 있는 폴더 경로만 바꿔주세요
    pipeline_load_folder(r"C:\Users\wngus\stock\sk_hynix3")




✅ 총 0건 삽입 완료


In [34]:
import os
import glob
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결 함수
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT")
    )

# 🔧 컬럼 표준화 목록
STANDARD_COLUMNS = [
    'ticker', 'price_date', 'publisher_name', 'title',
    'content', 'url', 'published_at'
]

# 📄 DataFrame 정규화 함수
def normalize_df(df: pd.DataFrame, ticker: str = None) -> pd.DataFrame:
    df = df.copy()
    # ticker 지정
    df['ticker'] = ticker
    # published_at 파싱
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['price_date'] = df['published_at'].dt.date
    # 부족한 컬럼 추가
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = None
    # 표준 순서 반환
    return df[STANDARD_COLUMNS]

# 🛠 발행처 조회/삽입 헬퍼
def get_or_create_publisher(cur, name: str) -> int:
    cur.execute(
        """
        WITH ins AS (
          INSERT INTO publisher (name)
          VALUES (%s)
          ON CONFLICT (name) DO NOTHING
          RETURNING publisher_id
        )
        SELECT publisher_id FROM ins
        UNION
        SELECT publisher_id FROM publisher WHERE name = %s;
        """,
        (name, name)
    )
    return cur.fetchone()[0]

# 🚀 메인 파이프라인: CSV 읽기 → 정규화 → DB 삽입
def pipeline_load_folder(folder_path: str):
    # CSV 파일 목록 로드
    files = glob.glob(os.path.join(folder_path, '*.csv'))
    print(f"🔍 Found {len(files)} CSV files in {folder_path}")
    for f in files:
        print(f"  - {f}")

    conn = get_db_connection()
    cur = conn.cursor()
    total_inserted = 0

    for file in files:
        # 파일명에서 ticker 추출 (e.g., sk_hynix.csv)
        fname = os.path.basename(file)
        parts = os.path.splitext(fname)[0].split('_')
        ticker = parts[-1] if parts[-1].isdigit() else None
        # fallback to default ticker
        if not ticker:
            ticker = '000660'  # SK하이닉스 티커 코드로 설정
            print(f"   → No ticker in filename, defaulting to {ticker}")

        # CSV 불러오기 및 디버그 정보 출력
        df_raw = pd.read_csv(file)
        print(f"▶ Reading {file}: {df_raw.shape[0]} rows, columns: {list(df_raw.columns)}")
        df = normalize_df(df_raw, ticker=ticker)
        print(f"   → After normalize: {df.shape[0]} rows, columns: {list(df.columns)}")

        for _, row in df.iterrows():
            # 단계별 디버그
            print(f"   Checking ticker: {row['ticker']}, price_date: {row['price_date']}")

            # ticker 유효성 검사
            cur.execute("SELECT 1 FROM ticker WHERE ticker = %s", (row['ticker'],))
            if cur.fetchone() is None:
                print("     → SKIP: ticker not found")
                continue

            # stock_price 유효성 검사
            cur.execute(
                "SELECT 1 FROM stock_price WHERE ticker = %s AND price_date = %s",
                (row['ticker'], row['price_date'])
            )
            if cur.fetchone() is None:
                print("     → SKIP: stock_price not found")
                continue

            # publisher 처리
            publisher_id = get_or_create_publisher(cur, row['publisher_name'] or 'Unknown')

            # news 삽입
            cur.execute(
                """
                INSERT INTO news (
                  ticker, price_date, publisher_id,
                  title, content, url, published_at
                ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                ON CONFLICT (url) DO NOTHING
                RETURNING news_id;
                """,
                (
                    row['ticker'], row['price_date'], publisher_id,
                    row['title'], row['content'], row['url'], row['published_at']
                )
            )
            result = cur.fetchone()
            if result:
                total_inserted += 1
                print(f"     → Inserted news_id: {result[0]}")
            else:
                print("     → SKIP: URL conflict (already exists)")

        # 파일 단위 커밋
        conn.commit()

    cur.close()
    conn.close()
    print(f"✅ 총 {total_inserted}건 삽입 완료")

if __name__ == '__main__':
    # CSV 파일들이 모여 있는 폴더 경로로 수정하세요
    pipeline_load_folder(r"C:\Users\wngus\stock\sk_hynix3.csv")

🔍 Found 0 CSV files in C:\Users\wngus\stock\sk_hynix3.csv
✅ 총 0건 삽입 완료


## 하이닉스 db 저장 코드

In [ ]:
import os
import glob
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# 🔐 환경 변수 로드
load_dotenv('stock.env')

# 📦 DB 연결 함수
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT")
    )

# 🔧 Normalize DataFrame to standard schema
STANDARD_COLUMNS = [
    'ticker', 'price_date', 'publisher_name', 'title',
    'content', 'url', 'published_at'
]

def normalize_df(df: pd.DataFrame, ticker: str = None) -> pd.DataFrame:
    df = df.copy()
    # ticker 채우기
    df['ticker'] = ticker
    # published_at 파싱
    df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
    df['price_date'] = df['published_at'].dt.date
    # 부족한 컬럼 추가
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = None
    # 순서 맞추기
    return df[STANDARD_COLUMNS]

# 🛠 Publisher helper
def get_or_create_publisher(cur, name: str) -> int:
    cur.execute(
        """
        WITH ins AS (
          INSERT INTO publisher (name)
          VALUES (%s)
          ON CONFLICT (name) DO NOTHING
          RETURNING publisher_id
        )
        SELECT publisher_id FROM ins
        UNION
        SELECT publisher_id FROM publisher WHERE name = %s;
        """,
        (name, name)
    )
    return cur.fetchone()[0]

# 🚀 Main pipeline: read CSVs, normalize, insert
def pipeline_load_folder(folder_path: str):
    conn = get_db_connection()
    cur = conn.cursor()
    total_inserted = 0

    # .csv 파일만 읽기
    files = glob.glob(os.path.join(folder_path, '*.csv'))
    for file in files:
        # 파일명에서 ticker 추출
        fname = os.path.basename(file)
        parts = os.path.splitext(fname)[0].split('_')
        ticker = parts[-1] if parts[-1].isdigit() else None

        df_raw = pd.read_csv(file)
        df = normalize_df(df_raw, ticker=ticker)

        for _, row in df.iterrows():
            try:
                # ticker 유효성
                cur.execute("SELECT 1 FROM ticker WHERE ticker = %s", (row['ticker'],))
                if cur.fetchone() is None:
                    continue

                # stock_price 유효성
                cur.execute(
                    "SELECT 1 FROM stock_price WHERE ticker = %s AND price_date = %s",
                    (row['ticker'], row['price_date'])
                )
                if cur.fetchone() is None:
                    continue

                # publisher
                publisher_id = get_or_create_publisher(cur, row['publisher_name'] or 'Unknown')

                # news 삽입
                cur.execute(
                    """
                    INSERT INTO news (
                      ticker, price_date, publisher_id,
                      title, content, url, published_at
                    )
                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (url) DO NOTHING RETURNING news_id;
                    """,
                    (
                        row['ticker'], row['price_date'], publisher_id,
                        row['title'], row['content'], row['url'],
                        row['published_at']
                    )
                )
                if cur.fetchone():
                    total_inserted += 1

            except Exception:
                conn.rollback()
                continue

        conn.commit()

    cur.close()
    conn.close()
    print(f"✅ 총 {total_inserted}건 삽입 완료")

if __name__ == '__main__':
    # csv 파일들이 모여 있는 폴더 경로만 바꿔주세요
    pipeline_load_folder(r"C:\Users\wngus\stock\sk_hynix3")




✅ 총 0건 삽입 완료


In [ ]:

import os
import pandas as pd
import psycopg2
from dotenv import load_dotenv

# 1) 환경 변수 로드
load_dotenv('stock.env')

# 2) DB 연결 함수
def get_db_connection():
    return psycopg2.connect(
        host=os.getenv("DB_HOST"),
        dbname=os.getenv("DB_NAME"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        port=os.getenv("DB_PORT")
    )

# 3) 발행처 조회/삽입 헬퍼
def get_or_create_publisher(cur, name: str) -> int:
    cur.execute("""
        WITH ins AS (
          INSERT INTO publisher (name)
          VALUES (%s)
          ON CONFLICT (name) DO NOTHING
          RETURNING publisher_id
        )
        SELECT publisher_id FROM ins
        UNION
        SELECT publisher_id FROM publisher WHERE name = %s;
    """, (name, name))
    return cur.fetchone()[0]

# 4) 단일 CSV 파일 파이프라인 함수
def pipeline_load_one(file_path: str):
    conn = get_db_connection()
    cur = conn.cursor()
    inserted = 0

    print(f"▶ Processing {file_path}")
    # CSV 로드 (문자열로 읽기)
    df = pd.read_csv(file_path, dtype=str)

    # ticker 정제
    df['ticker'] = df['ticker'].str.replace("'", "").str.strip()

    # 날짜 변환 (YYYYMMDD -> datetime)
    df['published_at'] = pd.to_datetime(df['published_at'], format='%Y%m%d', errors='coerce')
    df['price_date'] = df['published_at'].dt.date

    # content 컬럼 없으면 빈 문자열 생성
    if 'content' not in df.columns:
        df['content'] = ''

    # 컬럼명 통일: publisher -> publisher_name
    if 'publisher' in df.columns:
        df.rename(columns={'publisher': 'publisher_name'}, inplace=True)

    # 필수 컬럼 보장
    for col in ('ticker','publisher_name','title','url','summary','content','published_at','price_date'):
        if col not in df.columns:
            df[col] = None

    # 삽입 루프
    for _, row in df.iterrows():
        # ticker 존재 확인
        cur.execute("SELECT 1 FROM ticker WHERE ticker = %s", (row['ticker'],))
        if cur.fetchone() is None:
            continue

        # stock_price FK 대응: 없는 경우 더미 레코드 삽입
        cur.execute("""
            INSERT INTO stock_price (
              ticker, price_date,
              open_price, high_price, low_price,
              close_price, adj_close, volume
            ) VALUES (%s, %s, 0,0,0,0,0,0)
            ON CONFLICT (ticker, price_date) DO NOTHING;
        """, (row['ticker'], row['price_date']))

        # 발행처 처리
        pid = get_or_create_publisher(cur, row.get('publisher_name') or 'Unknown')

        # news 삽입
        cur.execute("""
            INSERT INTO news (
              ticker, price_date, publisher_id,
              title, summary, content, url, published_at
            ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (url) DO NOTHING
            RETURNING news_id;
        """, (
            row['ticker'], row['price_date'], pid,
            row['title'], row['summary'], row['content'],
            row['url'], row['published_at']
        ))
        if cur.fetchone():
            inserted += 1

    # 커밋 및 리소스 정리
    conn.commit()
    cur.close()
    conn.close()

    print(f"✅ 총 {inserted}건 삽입 완료")

# 5) 실행
if __name__ == '__main__':
    pipeline_load_one(r"C:\Users\wngus\stock\sk_hynix3.csv")


▶ Processing C:\Users\wngus\stock\sk_hynix3.csv
✅ 총 316건 삽입 완료
